# Módulo 5: Cálculo II
## Lección 5: Derivación de Campos Escalares y Optimización

La derivación de campos escalares es una herramienta fundamental en el cálculo multivariable que modela de manera exacta cómo varían magnitudes físicas en el espacio y en el tiempo. Fenómenos naturales complejos en física e ingeniería (tales como la propagación del calor, la distribución del potencial electrostático en el espacio o las tensiones en estructuras de ingeniería civil) se rigen por tasas de cambio continuo respecto a múltiples coordenadas espaciales.

En esta lección, exploraremos de manera formal el concepto de derivada parcial y direccional, la utilidad física del diferencial total, las cadenas de dependencia compuestas y la igualdad de derivadas mixtas cruzadas de Clairaut. Además, nos adentraremos en el modelado local mediante la fórmula de Taylor y formularemos un resolvedor automatizado para clasificar extremos locales utilizando los autovalores de la matriz Hessiana, aplicando la visualización 3D y la verificación simbólica en Python.

### Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Comprender la derivada direccional** de un campo escalar, su cálculo por teorema y su interpretación como pendiente en el espacio.
2. **Calcular derivadas parciales** en funciones de dos y tres variables e interpretarlas como pendientes en planos paralelos a los ejes.
3. **Aplicar el diferencial total** en física e ingeniería para aproximar linealmente cambios locales (ej. Ley de gases ideales e IMC/BMI).
4. **Resolver derivadas compuestas** usando la regla de la cadena multivariable.
5. **Calcular derivadas de orden superior** y comprobar la igualdad de derivadas mixtas cruzadas (Teorema de Clairaut).
6. **Aproximar superficies mediante polinomios de Taylor** de segundo orden en 3D.
7. **Clasificar puntos críticos** (máximos, mínimos y puntos de silla) evaluando los autovalores de la matriz Hessiana, validando tus resultados simbólicamente con `SymPy`.

### 1. Derivadas Parciales y Derivadas Direccionales

#### Derivadas Parciales:
Dada una función de dos variables $f(x, y)$, la derivada parcial con respecto a $x$ en el punto $(a, b)$ es la derivada ordinaria de la función respecto de $x$ tratando a $y$ como una constante:

$$f_x(a, b) = \lim_{h \to 0} \frac{f(a+h, b) - f(a, b)}{h}$$

De manera similar, la derivada parcial con respecto a $y$ es:

$$f_y(a, b) = \lim_{h \to 0} \frac{f(a, b+h) - f(a, b)}{h}$$

Geométricamente, $f_x(a, b)$ representa la pendiente de la recta tangente a la curva de intersección de la superficie $z = f(x, y)$ con el plano vertical $y = b$.

#### Derivadas Direccionales:
Para medir la tasa de cambio de la función en cualquier dirección dada por un vector unitario $\mathbf{u} = \langle a, b \rangle$, definimos la **derivada direccional**:

$$D_{\mathbf{u}} f(x_0, y_0) = \lim_{h \to 0} \frac{f(x_0 + ha, y_0 + hb) - f(x_0, y_0)}{h}$$

**Teorema:** Si $f$ es una función diferenciable, su derivada direccional en la dirección de $\mathbf{u} = \langle a, b \rangle$ se calcula como el producto escalar del vector gradiente y el vector unitario $\mathbf{u}$:

$$D_{\mathbf{u}} f(x, y) = \nabla f(x, y) \cdot \mathbf{u} = f_x(x, y)a + f_y(x, y)b$$

A continuación, graficaremos la superficie $f(x, y) = x^3 - 3xy + 4y^2$ y su recta tangente en la dirección $\theta = \pi/6$ en el punto $(1, 2)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from mpl_toolkits.mplot3d import Axes3D

# Configuración de estilo visual
plt.style.use('seaborn-v0_8-whitegrid')

def f_superficie(x, y):
    return x**3 - 3*x*y + 4*y**2

x0, y0 = 1.0, 2.0
z0 = f_superficie(x0, y0) # z0 = 1 - 6 + 16 = 11

# Derivadas parciales analíticas en (1, 2)
fx_val = 3*x0**2 - 3*y0  # -3
fy_val = -3*x0 + 8*y0   # 13

# Vector unitario en dirección theta = pi/6
theta = np.pi / 6.0
a, b = np.cos(theta), np.sin(theta)

# Derivada direccional
du_f = fx_val * a + fy_val * b

# Mallas para graficación
x = np.linspace(-1, 3, 50)
y = np.linspace(0, 4, 50)
X, Y = np.meshgrid(x, y)
Z = f_superficie(X, Y)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.6, edgecolor='none')

# Graficar punto P0
ax.scatter(x0, y0, z0, color='black', s=100, zorder=10, label=f'P0(1, 2, {z0})')

# Graficar recta tangente en la dirección u
t = np.linspace(-1, 1, 30)
rx = x0 + t * a
ry = y0 + t * b
rz = z0 + t * du_f
ax.plot(rx, ry, rz, 'r-', linewidth=3.5, label=f'Recta Tangente (Pendiente: {du_f:.3f})')

ax.set_title('Superficie y Recta Tangente de Derivada Direccional')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

### 2. Diferencial Total y Aproximación Lineal

Si $f(x, y)$ es una función diferenciable, su **diferencial total** $df$ representa la aproximación lineal del cambio en la función debido a variaciones pequeñas $dx$ y $dy$:

$$df = \frac{\partial f}{\partial x} dx + \frac{\partial f}{\partial y} dy$$

En física, la aproximación diferencial es sumamente útil para estimar incertidumbres experimentales o modelar variaciones aproximadas de variables de estado sin realizar el cálculo completo no lineal.

#### Caso de Estudio Físico: Ley de Gases Ideales
La presión $P(V, T)$ de un gas ideal está dada por:

$$P(V, T) = \frac{nRT}{V}$$

Tomando la diferencial total de la presión:

$$dP = \frac{\partial P}{\partial V} dV + \frac{\partial P}{\partial T} dT = -\frac{nRT}{V^2} dV + \frac{nR}{V} dT$$

Simularemos el comportamiento de la presión para $n=1\text{ mol}$, $R=8.314\text{ J/(mol K)}$ en el punto de volumen $V_0=1\text{ m}^3$ y temperatura $T_0=300\text{ K}$, comparando la aproximación lineal del diferencial total ($dP$) contra la diferencia exacta ($\Delta P$).

In [ ]:
# Parámetros de gas ideal
n = 1.0  # mol
R = 8.314  # J/(mol K)
V0 = 1.0   # m^3
T0 = 300.0 # K

# Presión inicial
P0 = (n * R * T0) / V0

# Variaciones
dV_vals = np.linspace(-0.1, 0.1, 50)  # cambio de volumen del -10% al 10%
dT = 5.0  # Cambio de temperatura fijo de +5 K

dP_approx = []
dP_exact = []

# Derivadas parciales en (V0, T0)
dP_dV = - (n * R * T0) / (V0**2)
dP_dT = (n * R) / V0

for dV in dV_vals:
    # Diferencial total
    dp_a = dP_dV * dV + dP_dT * dT
    dP_approx.append(dp_a)
    
    # Cambio exacto
    P_new = (n * R * (T0 + dT)) / (V0 + dV)
    dP_exact.append(P_new - P0)

# Graficar comparación
plt.figure(figsize=(8, 5))
plt.plot(dV_vals, dP_exact, 'k-', linewidth=2, label='Cambio Exacto $\Delta P$')
plt.plot(dV_vals, dP_approx, 'r--', linewidth=2, label='Diferencial Total $dP$ (Aprox. Lineal)')
plt.title('Diferencial Total vs. Cambio Real de Presión en Gas Ideal')
plt.xlabel('Variación de Volumen $dV$ ($m^3$)')
plt.ylabel('Cambio de Presión $dP$ ($Pa$)')
plt.legend(frameon=True)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 3. Regla de la Cadena y Teorema de Clairaut

#### Regla de la Cadena (Caso de 1 variable independiente intermedia):
Si $z = f(x, y)$ es diferenciable, y a su vez $x = g(t)$ e $y = h(t)$ son diferenciables respecto de $t$, la derivada de $z$ con respecto a $t$ está dada por:

$$\frac{dz}{dt} = \frac{\partial z}{\partial x} \frac{dx}{dt} + \frac{\partial z}{\partial y} \frac{dy}{dt}$$

#### Teorema de Clairaut (Derivadas Parciales Mixtas):
Si $f(x, y)$ está definida en un disco abierto $D \subset \mathbb{R}^2$ y las derivadas cruzadas de segundo orden $f_{xy}$ y $f_{yx}$ son continuas en $D$, entonces:

$$f_{xy}(a, b) = f_{yx}(a, b)$$

A continuación, calcularemos y demostraremos que se cumple el Teorema de Clairaut para las derivadas de segundo orden de la función:

$$f(x, y, z) = \sin(3x + yz)$$

In [ ]:
# Demostración simbólica de Clairaut para f(x,y,z) = sin(3x + yz)
x_sym, y_sym, z_sym = sp.symbols('x y z', real=True)
f_sym = sp.sin(3*x_sym + y_sym*z_sym)

# Derivadas de primer orden
fx = f_sym.diff(x_sym)
fy = f_sym.diff(y_sym)

# Derivadas cruzadas mixtas de segundo orden
fxy = fx.diff(y_sym)
fyx = fy.diff(x_sym)

print("Función escalar:")
sp.pprint(f_sym)
print("\nDerivada f_xy(x,y,z):")
sp.pprint(fxy)
print("\nDerivada f_yx(x,y,z):")
sp.pprint(fyx)

# Verificar la diferencia
diferencia = sp.simplify(fxy - fyx)
print(f"\nDiferencia simbólica f_xy - f_yx: {diferencia}")
assert diferencia == 0, "El teorema de Clairaut no se cumple."
print("¡Verificación exitosa: f_xy = f_yx por el Teorema de Clairaut!")

### 4. Fórmula de Taylor en Dos Variables

La fórmula de Taylor permite aproximar de manera local una función multivariable en el entorno de un punto $(x_0, y_0)$ mediante un polinomio de grado $k$. Para una aproximación cuadrática ($k=2$), la expresión es:

$$f(x, y) \approx f(x_0, y_0) + \left[f_x(x_0, y_0)(x - x_0) + f_y(x_0, y_0)(y - y_0)\right] + \frac{1}{2} \left[f_{xx}(x_0, y_0)(x - x_0)^2 + 2f_{xy}(x_0, y_0)(x-x_0)(y-y_0) + f_{yy}(x_0, y_0)(y-y_0)^2\right]$$

Si aproximamos la función $f(x, y) = e^{xy}$ en el origen $(0, 0)$:
- $f(0, 0) = 1$
- $f_x(0,0) = 0$, $f_y(0,0) = 0$
- $f_{xx}(0,0) = 0$, $f_{yy}(0,0) = 0$, $f_{xy}(0,0) = 1$

La aproximación cuadrática local en el origen es:

$$P_2(x, y) = 1 + xy$$

A continuación, graficaremos la superficie real de $f(x, y) = e^{xy}$ frente a su paraboloide de Taylor aproximante $P_2(x, y)$.

In [ ]:
x_vals = np.linspace(-1.5, 1.5, 50)
y_vals = np.linspace(-1.5, 1.5, 50)
X_m, Y_m = np.meshgrid(x_vals, y_vals)

Z_real = np.exp(X_m * Y_m)
Z_approx = 1.0 + X_m * Y_m

fig = plt.figure(figsize=(12, 5))

# 1. Superficie Real vs Aproximada
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X_m, Y_m, Z_real, cmap='Blues', alpha=0.5, edgecolor='none', label='Real: $e^{xy}$')
ax1.plot_surface(X_m, Y_m, Z_approx, cmap='Oranges', alpha=0.5, edgecolor='none', label='Aprox: $1+xy$')
ax1.set_title('Superficie Real vs. Aproximación de Taylor')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')

# 2. Error absoluto local
ax2 = fig.add_subplot(122, projection='3d')
error_surf = ax2.plot_surface(X_m, Y_m, np.abs(Z_real - Z_approx), cmap='inferno', edgecolor='none')
fig.colorbar(error_surf, ax=ax2, label='Error Absoluto')
ax2.set_title('Error Absoluto de la Aproximación')
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Error')

plt.tight_layout()
plt.show()

### 5. Puntos Estacionarios y Criterio de la Matriz Hessiana

Para un campo escalar diferenciable $f: \mathbb{R}^n \to \mathbb{R}$, los **puntos críticos (o estacionarios)** son aquellos donde el vector gradiente es nulo:

$$\nabla f(\mathbf{x}) = \mathbf{0}$$

Para determinar la naturaleza de estos puntos, construimos la **matriz Hessiana** $H_f$, que contiene las segundas derivadas parciales de la función:

$$H_f(x, y) = \begin{pmatrix} f_{xx} & f_{xy} \\ f_{yx} & f_{yy} \end{pmatrix}$$

#### Criterio de los Valores Propios (Autovalores):
Evaluando la matriz Hessiana en un punto crítico $\mathbf{x}_0$, se analizan sus valores propios (autovalores) $\lambda_i$:
1. **Mínimo local:** Todos los valores propios son estrictamente positivos (Hessiana es definida positiva).
2. **Máximo local:** Todos los valores propios son estrictamente negativos (Hessiana es definida negativa).
3. **Punto de silla (saddle point):** Hay valores propios positivos y negativos de manera simultánea (Hessiana es indefinida).
4. **Criterio no concluyente:** Al menos un valor propio es cero (el determinante es nulo), requiriendo análisis locales de mayor orden.

A continuación, implementaremos un resolvedor automatizado que evalúe y clasifique puntos críticos para cualquier superficie bidimensional.

### 6. Estudio de Caso: Superficie de Múltiples Extremos

Analizaremos la función bidimensional:

$$f(x, y) = x^4 - 4x^2 + y^4 - 4y^2$$

#### Análisis Analítico:
1. **Gradiente:**
   $$\nabla f(x, y) = \langle 4x^3 - 8x, \quad 4y^3 - 8y \rangle$$
   
2. **Puntos Críticos:** Resolviendo $4x(x^2 - 2) = 0 \implies x \in \{0, \sqrt{2}, -\sqrt{2}\}$ y $4y(y^2 - 2) = 0 \implies y \in \{0, \sqrt{2}, -\sqrt{2}\}$. Esto da $3 \times 3 = 9$ puntos críticos en total.
3. **Segundas Derivadas:** $f_{xx} = 12x^2 - 8$, $f_{yy} = 12y^2 - 8$, $f_{xy} = 0$.
4. **Hessiana:**
   $$H_f(x, y) = \begin{pmatrix} 12x^2 - 8 & 0 \\ 0 & 12y^2 - 8 \end{pmatrix}$$

Clasificaremos numéricamente los 9 puntos críticos y los graficaremos en 3D sobre la superficie utilizando marcadores de colores específicos para representar mínimos locales (rojo), máximos locales (azul) y puntos de silla (verde).

In [ ]:
def f_extremos(x, y):
    return x**4 - 4*x**2 + y**4 - 4*y**2

# Definición analítica de los puntos críticos
raices = [0.0, np.sqrt(2), -np.sqrt(2)]
puntos_criticos = []
for rx in raices:
    for ry in raices:
        puntos_criticos.append((rx, ry))

# Clasificación
min_list = []
max_list = []
saddle_list = []

for (px, py) in puntos_criticos:
    # Evaluar componentes de la Hessiana diagonal
    h11 = 12 * px**2 - 8
    h22 = 12 * py**2 - 8
    
    val_z = f_extremos(px, py)
    
    if h11 > 0 and h22 > 0:
        min_list.append((px, py, val_z))
    elif h11 < 0 and h22 < 0:
        max_list.append((px, py, val_z))
    else:
        saddle_list.append((px, py, val_z))

print("CLASIFICACIÓN DE PUNTOS CRÍTICOS:")
print(f"  - Mínimos locales ({len(min_list)}): {min_list}")
print(f"  - Máximos locales ({len(max_list)}): {max_list}")
print(f"  - Puntos de silla ({len(saddle_list)}): {saddle_list}")

# Graficación
xg = np.linspace(-2.2, 2.2, 100)
yg = np.linspace(-2.2, 2.2, 100)
XG, YG = np.meshgrid(xg, yg)
ZG = f_extremos(XG, YG)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(XG, YG, ZG, cmap='coolwarm', alpha=0.5, edgecolor='none')

# Marcadores
if min_list:
    mx, my, mz = zip(*min_list)
    ax.scatter(mx, my, mz, color='red', s=130, label='Mínimos Locales', depthshade=False, zorder=15)
if max_list:
    maxx, maxy, maxz = zip(*max_list)
    ax.scatter(maxx, maxy, maxz, color='blue', s=130, label='Máximo Local', depthshade=False, zorder=15)
if saddle_list:
    sx, sy, sz = zip(*saddle_list)
    ax.scatter(sx, sy, sz, color='green', s=130, label='Puntos de Silla', depthshade=False, zorder=15)

ax.set_title('Extremos Relativos en la Superficie $f(x, y) = x^4 - 4x^2 + y^4 - 4y^2$', fontsize=12, fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()

### 7. Capa de Verificación Simbólica con SymPy

Para asegurar la consistencia matemática y corroborar la precisión de los autovalores y la naturaleza de los extremos locales, utilizaremos `SymPy` para obtener la matriz Hessiana formal de nuestra función y calcular los autovalores en el origen de manera exacta.

In [ ]:
x_sym, y_sym = sp.symbols('x y', real=True)
f_sym = x_sym**4 - 4*x_sym**2 + y_sym**4 - 4*y_sym**2

# Obtener gradiente y Hessiana simbólica
gradiente_sym = [f_sym.diff(x_sym), f_sym.diff(y_sym)]
hessiana_sym = sp.hessian(f_sym, (x_sym, y_sym))

# Resolver de forma exacta para gradiente = 0
puntos_criticos_exactos = sp.solve(gradiente_sym, (x_sym, y_sym))

print("Gradiente de f(x, y):")
sp.pprint(gradiente_sym)
print("\nMatriz Hessiana general de f(x, y):")
sp.pprint(hessiana_sym)

# Evaluar en el origen (0, 0)
hess_origen = hessiana_sym.subs({x_sym: 0, y_sym: 0})
autovalores_origen = list(hess_origen.eigenvals().keys())

print("\nMatriz Hessiana en (0, 0):")
sp.pprint(hess_origen)
print(f"\nAutovalores en (0, 0): {autovalores_origen}")

# Verificación analítica
assert all(val < 0 for val in autovalores_origen), "El origen debe poseer autovalores negativos (máximo)."
print("\n¡Verificación simbólica de autovalores de la matriz Hessiana aprobada con éxito!")

### Resumen de Conceptos Clave

1. **Derivadas Parciales y Direccionales:** Describen la tasa de cambio local de un campo escalar. El gradiente apunta en la dirección de máximo crecimiento y su producto escalar con la dirección unitaria da la derivada direccional.
2. **Diferencial Total:** Proporciona la aproximación lineal local del cambio total de una función multivariable debido a variaciones arbitrarias de sus variables independientes.
3. **Teorema de Clairaut:** Establece la simetría de las segundas derivadas parciales cruzadas continuas ($f_{xy} = f_{yx}$).
4. **Fórmula de Taylor:** Permite aproximar localmente funciones no lineales por polinomios de mayor grado. Para segundo orden, la aproximación es un paraboloide local.
5. **Matriz Hessiana y Extremos:** Un punto crítico es un punto estacionario del gradiente. El criterio de los autovalores de la matriz Hessiana permite clasificarlos en máximos (todos $\lambda < 0$), mínimos (todos $\lambda > 0$) y puntos de silla (signos mixtos).

### Referencias Bibliográficas

- Marsden, J. E., & Tromba, A. J. (2012). *Cálculo Vectorial* (6ta ed.). Pearson.
- Thomas, G. B. (2014). *Cálculo: Varias Variables* (13va ed.). Pearson.
- Apostol, T. M. (1969). *Calculus: Vol. 2* (2da ed.). John Wiley & Sons.